In [1]:
import os
from pathlib import Path

def get_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "data" / "raw").exists() or (candidate / "requirements.txt").exists():
            return candidate
    return current

PROJECT_ROOT = get_project_root()
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import os

# Safe Colab import for cross-environment portability
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    files = None

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
import os
from pathlib import Path
import pandas as pd

# Load dataset using robust local workspace or Colab path
raw_file = PROJECT_ROOT / "data" / "raw" / "Student Performance Factors.csv"
if not raw_file.exists() and os.path.exists("/content/Student Performance Factors.csv"):
    raw_file = Path("/content/Student Performance Factors.csv")

df = pd.read_csv(raw_file)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
display(df.head())


Dataset loaded successfully.
Dataset shape: (6607, 20)


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [3]:
# Outlier Correction: Cap Exam_Score > 100 to 100 before splitting (identified in EDA)
outliers_count = (df["Exam_Score"] > 100).sum()
if outliers_count > 0:
    df.loc[df["Exam_Score"] > 100, "Exam_Score"] = 100
    print(f"Adjusted {outliers_count} record(s) where Exam_Score > 100 to 100.")

target = "Exam_Score"
X = df.drop(columns=[target])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


Adjusted 1 record(s) where Exam_Score > 100 to 100.
Feature shape: (6607, 19)
Target shape: (6607,)


In [4]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']

Categorical features:
['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


C:\Users\mansi\AppData\Local\Temp\ipykernel_7116\1998881318.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [5]:
X_clean = X.copy()

X_clean[categorical_features] = (
    X_clean[categorical_features]
    .fillna("Missing")
)

print("Total missing values after cleaning:")
print(X_clean.isnull().sum().sum())

Total missing values after cleaning:
0


In [6]:
cleaned_dataset = pd.concat(
    [X_clean, y],
    axis=1
)

print("Dataset cleaned and target reattached in memory. Shape:", cleaned_dataset.shape)


Dataset cleaned and target reattached in memory. Shape: (6607, 20)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 5285
Testing samples: 1322


In [8]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [10]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


Processed training shape: (5285, 43)
Processed testing shape: (1322, 43)


In [11]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

print("\nFirst 20 processed features:")
print(feature_names[:20])

Number of processed features: 43

First 20 processed features:
['num__Hours_Studied' 'num__Attendance' 'num__Sleep_Hours'
 'num__Previous_Scores' 'num__Tutoring_Sessions' 'num__Physical_Activity'
 'cat__Parental_Involvement_High' 'cat__Parental_Involvement_Low'
 'cat__Parental_Involvement_Medium' 'cat__Access_to_Resources_High'
 'cat__Access_to_Resources_Low' 'cat__Access_to_Resources_Medium'
 'cat__Extracurricular_Activities_No'
 'cat__Extracurricular_Activities_Yes' 'cat__Motivation_Level_High'
 'cat__Motivation_Level_Low' 'cat__Motivation_Level_Medium'
 'cat__Internet_Access_No' 'cat__Internet_Access_Yes'
 'cat__Family_Income_High']


In [12]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

display(X_train_processed_df.head())

,num__Hours_Studied,num__Attendance,num__Sleep_Hours,num__Previous_Scores,num__Tutoring_Sessions,num__Physical_Activity,cat__Parental_Involvement_High,cat__Parental_Involvement_Low,cat__Parental_Involvement_Medium,cat__Access_to_Resources_High,cat__Access_to_Resources_Low,cat__Access_to_Resources_Medium,cat__Extracurricular_Activities_No,cat__Extracurricular_Activities_Yes,cat__Motivation_Level_High,cat__Motivation_Level_Low,cat__Motivation_Level_Medium,cat__Internet_Access_No,cat__Internet_Access_Yes,cat__Family_Income_High,cat__Family_Income_Low,cat__Family_Income_Medium,cat__Teacher_Quality_High,cat__Teacher_Quality_Low,cat__Teacher_Quality_Medium,cat__Teacher_Quality_Missing,cat__School_Type_Private,cat__School_Type_Public,cat__Peer_Influence_Negative,cat__Peer_Influence_Neutral,cat__Peer_Influence_Positive,cat__Learning_Disabilities_No,cat__Learning_Disabilities_Yes,cat__Parental_Education_Level_College,cat__Parental_Education_Level_High School,cat__Parental_Education_Level_Missing,cat__Parental_Education_Level_Postgraduate,cat__Distance_from_Home_Far,cat__Distance_from_Home_Missing,cat__Distance_from_Home_Moderate,cat__Distance_from_Home_Near,cat__Gender_Female,cat__Gender_Male
5810,1.173579,-0.075846,0.669094,-0.837277,0.405784,1.964909,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1268,-0.661983,0.531894,-0.016740,1.314728,0.405784,0.030381,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
414,0.339232,0.618714,0.669094,0.551113,-0.405017,-1.904146,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4745,-0.328245,1.747376,2.040764,0.759372,-0.405017,0.030381,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
654,2.508533,-0.162667,2.040764,1.661825,-0.405017,-0.936882,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [13]:
train_processed = pd.concat(
    [
        X_train_processed_df,
        y_train.rename(target)
    ],
    axis=1
)

test_processed = pd.concat(
    [
        X_test_processed_df,
        y_test.rename(target)
    ],
    axis=1
)

print("Training dataset:")
display(train_processed.head())

print("\nTesting dataset:")
display(test_processed.head())

Training dataset:


,num__Hours_Studied,num__Attendance,num__Sleep_Hours,num__Previous_Scores,num__Tutoring_Sessions,num__Physical_Activity,cat__Parental_Involvement_High,cat__Parental_Involvement_Low,cat__Parental_Involvement_Medium,cat__Access_to_Resources_High,cat__Access_to_Resources_Low,cat__Access_to_Resources_Medium,cat__Extracurricular_Activities_No,cat__Extracurricular_Activities_Yes,cat__Motivation_Level_High,cat__Motivation_Level_Low,cat__Motivation_Level_Medium,cat__Internet_Access_No,cat__Internet_Access_Yes,cat__Family_Income_High,cat__Family_Income_Low,cat__Family_Income_Medium,cat__Teacher_Quality_High,cat__Teacher_Quality_Low,cat__Teacher_Quality_Medium,cat__Teacher_Quality_Missing,cat__School_Type_Private,cat__School_Type_Public,cat__Peer_Influence_Negative,cat__Peer_Influence_Neutral,cat__Peer_Influence_Positive,cat__Learning_Disabilities_No,cat__Learning_Disabilities_Yes,cat__Parental_Education_Level_College,cat__Parental_Education_Level_High School,cat__Parental_Education_Level_Missing,cat__Parental_Education_Level_Postgraduate,cat__Distance_from_Home_Far,cat__Distance_from_Home_Missing,cat__Distance_from_Home_Moderate,cat__Distance_from_Home_Near,cat__Gender_Female,cat__Gender_Male,Exam_Score
5810,1.173579,-0.075846,0.669094,-0.837277,0.405784,1.964909,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,69
1268,-0.661983,0.531894,-0.016740,1.314728,0.405784,0.030381,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,69
414,0.339232,0.618714,0.669094,0.551113,-0.405017,-1.904146,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,66
4745,-0.328245,1.747376,2.040764,0.759372,-0.405017,0.030381,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,72
654,2.508533,-0.162667,2.040764,1.661825,-0.405017,-0.936882,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,72



Testing dataset:


,num__Hours_Studied,num__Attendance,num__Sleep_Hours,num__Previous_Scores,num__Tutoring_Sessions,num__Physical_Activity,cat__Parental_Involvement_High,cat__Parental_Involvement_Low,cat__Parental_Involvement_Medium,cat__Access_to_Resources_High,cat__Access_to_Resources_Low,cat__Access_to_Resources_Medium,cat__Extracurricular_Activities_No,cat__Extracurricular_Activities_Yes,cat__Motivation_Level_High,cat__Motivation_Level_Low,cat__Motivation_Level_Medium,cat__Internet_Access_No,cat__Internet_Access_Yes,cat__Family_Income_High,cat__Family_Income_Low,cat__Family_Income_Medium,cat__Teacher_Quality_High,cat__Teacher_Quality_Low,cat__Teacher_Quality_Medium,cat__Teacher_Quality_Missing,cat__School_Type_Private,cat__School_Type_Public,cat__Peer_Influence_Negative,cat__Peer_Influence_Neutral,cat__Peer_Influence_Positive,cat__Learning_Disabilities_No,cat__Learning_Disabilities_Yes,cat__Parental_Education_Level_College,cat__Parental_Education_Level_High School,cat__Parental_Education_Level_Missing,cat__Parental_Education_Level_Postgraduate,cat__Distance_from_Home_Far,cat__Distance_from_Home_Missing,cat__Distance_from_Home_Moderate,cat__Distance_from_Home_Near,cat__Gender_Female,cat__Gender_Male,Exam_Score
743,0.005494,-0.770407,-0.016740,0.828791,-0.405017,1.964909,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,65
5551,0.339232,-0.770407,-0.016740,1.592406,0.405784,-0.936882,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,65
3442,0.172363,0.965995,-0.702575,-1.531472,-0.405017,0.030381,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,71
6571,-1.329460,0.965995,0.669094,0.412274,-1.215819,0.997645,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,64
4204,0.172363,-1.464968,0.669094,1.384147,0.405784,1.964909,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,66


In [14]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

train_processed.to_csv(
    processed_dir / "AcademiQ_Train_Preprocessed.csv",
    index=False
)

test_processed.to_csv(
    processed_dir / "AcademiQ_Test_Preprocessed.csv",
    index=False
)

print("Saved successfully:")
print("1. data/processed/AcademiQ_Train_Preprocessed.csv")
print("2. data/processed/AcademiQ_Test_Preprocessed.csv")


Saved successfully:
1. data/processed/AcademiQ_Train_Preprocessed.csv
2. data/processed/AcademiQ_Test_Preprocessed.csv


In [15]:
print("========== FINAL CHECK ==========")

print("\nOriginal dataset:")
print(df.shape)

print("\nCleaned dataset:")
print(cleaned_dataset.shape)

print("\nTraining dataset:")
print(train_processed.shape)

print("\nTesting dataset:")
print(test_processed.shape)

print("\nMissing values in training data:")
print(train_processed.isnull().sum().sum())

print("\nMissing values in testing data:")
print(test_processed.isnull().sum().sum())

========== FINAL CHECK ==========

Original dataset:
(6607, 20)

Cleaned dataset:
(6607, 20)

Training dataset:
(5285, 44)

Testing dataset:
(1322, 44)

Missing values in training data:
0

Missing values in testing data:
0


In [16]:
# Save cleaned training and testing data before encoding/scaling

train_cleaned = pd.concat(
    [X_train, y_train.rename(target)],
    axis=1
)

test_cleaned = pd.concat(
    [X_test, y_test.rename(target)],
    axis=1
)

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

train_cleaned.to_csv(
    processed_dir / "AcademiQ_Train_Cleaned.csv",
    index=False
)

test_cleaned.to_csv(
    processed_dir / "AcademiQ_Test_Cleaned.csv",
    index=False
)

print("Saved cleaned train/test datasets successfully to data/processed/.")


Saved cleaned train/test datasets successfully to data/processed/.


In [17]:
print("Training cleaned shape:", train_cleaned.shape)
print("Testing cleaned shape:", test_cleaned.shape)

print("\nSaved files in data/processed/:")
print("1. AcademiQ_Train_Cleaned.csv")
print("2. AcademiQ_Test_Cleaned.csv")
print("3. AcademiQ_Train_Preprocessed.csv")
print("4. AcademiQ_Test_Preprocessed.csv")


Training cleaned shape: (5285, 20)
Testing cleaned shape: (1322, 20)

Saved files in data/processed/:
1. AcademiQ_Train_Cleaned.csv
2. AcademiQ_Test_Cleaned.csv
3. AcademiQ_Train_Preprocessed.csv
4. AcademiQ_Test_Preprocessed.csv


In [18]:
if IN_COLAB and files is not None:
    for f in ["AcademiQ_Train_Cleaned.csv", "AcademiQ_Test_Cleaned.csv",
              "AcademiQ_Train_Preprocessed.csv", "AcademiQ_Test_Preprocessed.csv"]:
        try:
            target_path = PROJECT_ROOT / "data" / "processed" / f
            if target_path.exists():
                files.download(str(target_path))
        except Exception as e:
            print(f"Colab download skipped for {f}: {e}")
else:
    print("Preprocessed and cleaned datasets are saved locally in data/processed/.")


Preprocessed and cleaned datasets are saved locally in data/processed/.
